In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


def _edge_records(graph):
    from topologicpy.TGraph import TGraph
    records = TGraph.Edges(graph)
    if any(record.get("representation") is None for record in records):
        geometry = TGraph.Edges(graph, asTopologic=True, segmentCurves=False)
        if len(geometry) != len(records):
            raise ValueError("Could not generate one geometry representation per graph edge")
        for record, edge in zip(records, geometry):
            record["representation"] = edge
    return records


In [ ]:
# This is only needed on the software development machine. If you are a user, you can skip this step.
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")
# Make sure you have pip installed neo4j
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Aperture import Aperture
from topologicpy.TGraph import TGraph
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Plotly import Plotly



In [ ]:
def StreetSegmentEdgesByOSMPath(osmPath: str,
                               project=None,
                               useWebMercatorIfNone: bool = True,
                               mantissa: int = 6,
                               tolerance: float = 0.0001,
                               silent: bool = False):
    """
    OSM (.osm XML) -> graph-ready TopologicPy Edge segments with dictionaries.

    Filter:
    - EXCLUDE: cycleway, service, steps
    - INCLUDE: roads + footways only

    Graph-ready:
    - Returns flat list of Edge segments
    - Interns Vertices by (x,y,z) rounded to mantissa => shared endpoints become identical objects
    - Each segment has street_id (OSM way id) and street_name, plus segment_id/index

    Projection:
    - If project is None and useWebMercatorIfNone=True, lon/lat are projected to EPSG:3857
      (meters-ish). This avoids the "skewed" look from degree coordinates.
    """

    import math
    import xml.etree.ElementTree as ET

    from topologicpy.Vertex import Vertex
    from topologicpy.Edge import Edge
    from topologicpy.Dictionary import Dictionary
    from topologicpy.Topology import Topology

    # --- highway filters ---
    ROAD_HIGHWAYS = {
        "motorway", "motorway_link",
        "trunk", "trunk_link",
        "primary", "primary_link",
        "secondary", "secondary_link",
        "tertiary", "tertiary_link",
        "unclassified",
        "residential",
        "living_street",
        "road"
    }
    FOOTWAY_HIGHWAYS = {"footway", "pedestrian", "path"}
    EXCLUDE_HIGHWAYS = {"cycleway", "service", "steps"}

    def _round(x):
        return round(float(x), mantissa)

    def _dict_from_map(dmap: dict):
        keys = list(dmap.keys())
        vals = ["" if dmap[k] is None else str(dmap[k]) for k in keys]
        return Dictionary.ByKeysValues(keys, vals)

    # --- Default projection: lon/lat -> Web Mercator (EPSG:3857) ---
    # This is the standard slippy-map projection used by OSM tiles.
    # Units are meters (approximately), suitable for planar plotting/graph metrics.
    R = 6378137.0
    def _web_mercator(lon_deg, lat_deg):
        # clamp latitude to avoid infinity at poles
        lat = max(min(lat_deg, 85.05112878), -85.05112878)
        lon = lon_deg
        x = R * math.radians(lon)
        y = R * math.log(math.tan(math.pi / 4.0 + math.radians(lat) / 2.0))
        return x, y, 0.0

    # Node id -> (lon, lat)
    nodes = {}

    # Vertex interning cache: (x,y,z) -> Vertex
    vcache = {}

    def _vertex_by_xyz(x, y, z=0.0):
        k = (_round(x), _round(y), _round(z))
        v = vcache.get(k)
        if v is None:
            v = Vertex.ByCoordinates(_as_topology(k[0]), _as_topology(k[1]), _as_topology(k[2]))
            vcache[k] = v
        return v

    edges_out = []

    context = ET.iterparse(osmPath, events=("start", "end"))

    in_way = False
    way_id = None
    way_node_refs = None
    way_tags = None

    for event, elem in context:
        tag = elem.tag

        # ---- Nodes ----
        if event == "end" and tag == "node":
            nid = elem.attrib.get("id")
            lat = elem.attrib.get("lat")
            lon = elem.attrib.get("lon")
            if nid is not None and lat is not None and lon is not None:
                nodes[nid] = (float(lon), float(lat))
            elem.clear()

        # ---- Ways ----
        elif event == "start" and tag == "way":
            in_way = True
            way_id = elem.attrib.get("id")
            way_node_refs = []
            way_tags = {}

        elif in_way and event == "end" and tag == "nd":
            ref = elem.attrib.get("ref")
            if ref is not None:
                way_node_refs.append(ref)
            elem.clear()

        elif in_way and event == "end" and tag == "tag":
            k = elem.attrib.get("k")
            v = elem.attrib.get("v")
            if k is not None:
                way_tags[k] = v
            elem.clear()

        elif event == "end" and tag == "way":
            in_way = False

            highway = way_tags.get("highway", None)
            if highway is None:
                elem.clear()
                continue

            # keep roads + footways; exclude cycleway/service/steps
            if highway in EXCLUDE_HIGHWAYS:
                elem.clear()
                continue
            if (highway not in ROAD_HIGHWAYS) and (highway not in FOOTWAY_HIGHWAYS):
                elem.clear()
                continue

            # Build vertices for the way (skipping missing nodes)
            verts = []
            missing = 0
            for nid in way_node_refs:
                ll = nodes.get(nid)
                if ll is None:
                    missing += 1
                    continue
                lon, lat = ll

                if project is not None:
                    p = project(lon, lat)
                    if len(p) == 2:
                        x, y, z = p[0], p[1], 0.0
                    else:
                        x, y, z = p[0], p[1], p[2]
                else:
                    if useWebMercatorIfNone:
                        x, y, z = _web_mercator(lon, lat)
                    else:
                        # raw degrees (will look skewed in planar plotting)
                        x, y, z = lon, lat, 0.0

                verts.append(_vertex_by_xyz(x, y, z))

            if len(verts) < 2:
                elem.clear()
                continue

            # street grouping fields
            street_name = way_tags.get("name", "") or ""
            if (not street_name) and way_tags.get("ref"):
                street_name = way_tags.get("ref", "")

            base = dict(way_tags)
            base["osm_element"] = "way"
            base["osm_id"] = str(way_id) if way_id is not None else ""
            base["street_id"] = base["osm_id"]     # grouping key
            base["street_name"] = str(street_name)
            base["highway"] = str(highway)
            base["node_count"] = str(len(way_node_refs))
            base["missing_nodes"] = str(missing)
            base["crs"] = "EPSG:3857" if (project is None and useWebMercatorIfNone) else "custom_or_degrees"

            # segments
            for i in range(len(verts) - 1):
                e = Edge.ByStartVertexEndVertex(_as_topology(verts[i]), _as_topology(verts[i + 1]), tolerance=tolerance)

                seg = dict(base)
                seg["segment_index"] = str(i)
                seg["segment_id"] = f"{seg['street_id']}_{i}"

                Topology.SetDictionary(e, _dict_from_map(seg))
                edges_out.append(e)

            elem.clear()

    return edges_out


# Example with your uploaded file:
# edges = StreetSegmentEdgesByOSMPath("/mnt/data/map")


In [ ]:
# Example (your uploaded file path):
edges = StreetSegmentEdgesByOSMPath(r"C:\Users\sarwj\Downloads\map")
print(len(edges))
d = _record_dictionary(edges[0])
print(Dictionary.Keys(d))
print(Dictionary.ValueAtKey(d, "street_id"), Dictionary.ValueAtKey(d, "street_name"))
#Topology.Show(edges, backgroundColor="white", projection="ortho", camera=[0,0,100])

In [ ]:
cluster = Cluster.ByTopologies(_as_topology(edges))
verts = Topology.Vertices(_as_topology(cluster))
print(len(verts))

In [ ]:
g = TGraph.ByVerticesEdges(verts, edges)
#Topology.Show(g, backgroundColor="white", projection="ortho", camera=[0,0,100])

In [ ]:
lg = TGraph.LineGraph(g)
#Topology.Show(lg, backgroundColor="white", projection="ortho", camera=[0,0,100])

In [ ]:
g_verts = TGraph.Vertices(lg)
for i, v in enumerate(g_verts):
    d = _record_dictionary(v)
    if Dictionary.ValueAtKey(d, "street_name") == "":
        print(Dictionary.ValueAtKey(d, "street_id"))
        d = Dictionary.SetValueAtKey(d, "street_name", str(Dictionary.ValueAtKey(d, "street_id")))
        v = Topology.SetDictionary(v, d)
print(Dictionary.ValueAtKey(d, "street_name"))
#lg = Graph.RemoveVertex(lg, remove_vertices)

In [ ]:
qg = TGraph.Quotient(lg, key="street_name", transferDictionaries=True)

In [ ]:
Topology.Show(qg, backgroundColor="white", projection="ortho", camera=[0,0,100])

In [ ]:
qg_verts = TGraph.Vertices(qg)
for v in qg_verts:
    d = _record_dictionary(v)
    print(Dictionary.ValueAtKey(d, "street_name"))

In [ ]:
centralities = TGraph.ClosenessCentrality(qg, normalize=True)

In [ ]:
qg_verts = TGraph.Vertices(qg)
centralities = {}
for v in qg_verts:
    d = _record_dictionary(v)
    key = Dictionary.ValueAtKey(d, "street_name")
    value = Dictionary.ValueAtKey(d, "closeness_centrality")
    centralities[key] = value
    print(Dictionary.ValueAtKey(d, "street_name"),",",Dictionary.ValueAtKey(d, "closeness_centrality"))

In [ ]:
print(list(centralities))

In [ ]:
from topologicpy.Color import Color
edges = _edge_records(g)
for edge in edges:
    d = _record_dictionary(edge)
    key = Dictionary.ValueAtKey(d, "street_name")
    centrality = centralities.get(key, 0)
    color = Color.ByValueInRange(centrality, minValue=0.5, maxValue=1)
    d = Dictionary.SetValuesAtKeys(d, ["closeness_centrality", "cc_color"], [centrality, color])
    edge = Topology.SetDictionary(edge, d)

In [ ]:
Topology.Show(g, vertexSize=0.0001, edgeWidth=10, backgroundColor="black", projection="ortho", camera=[0,0,100], edgeColorKey="cc_color", width=1200, height=1200, renderer="browser")